In [ ]:
# BRONZE LAYER

import logging
from datetime import datetime
from pyspark.sql import functions as F
from delta.tables import DeltaTable

logging.basicConfig(level = logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger(__name__)

# PHASE 1: CONFIG
bronze_config = {
    "table_bronze" : "workspace.default.bronze_transactions",
    "source_file" : "/Volumes/workspace/default/raw_volume/transactions_batch_1/",
    "checkpoint_path" : "/Volumes/workspace/default/checkpoints/bronze_transactions",
    "batch_id_bronze" : f"BRONZE_{datetime.now().strftime('%Y%m%d_%H%M%S')}",
    "schema" : "txn_id STRING, cust_id LONG, amount DOUBLE, is_member BOOLEAN, points INT, status STRING, txn_date STRING",
    "repartition_num" : 5,
    "log_table" : {
        "batch" : "workspace.default.batch_control",
        "sla" : "workspace.default.sla_metrics"
    }
}

# PHASE 2: HELPER UTILS
def upsert_log(spark, table_name, schema, data, keys):
    df = spark.createDataFrame(data, schema)
    if not spark.catalog.tableExists(table_name):
        df.write.format("delta").mode("overwrite").saveAsTable(table_name)

    else:
        dt = DeltaTable.forName(spark, table_name)
        condition = " AND ".join([f"t.{k} = s.{k}" for k in keys])
        (dt.alias("t").merge(df.alias("s"), condition)
         .whenMatchedUpdateAll()
         .whenNotMatchedInsertAll()
         .execute())
        
def audit_col(df, batch_id):
    src_col = F.col("_metadata.file_path") if "_metadata" in df.columns else F.lit(None).cast("string")
    is_corrupt = F.when(F.col("_corrupt_record").isNotNull(), F.lit("CORRUPT")).otherwise(F.lit("CLEAN"))

    return (df.withColumn("_source_file", src_col)
              .withColumn("_ingest_at", F.current_timestamp())
              .withColumn("_batch_id_bronze", F.lit(batch_id))
              .withColumn("_ingest_date", F.current_date())
              .withColumn("_record_status", is_corrupt)
              .drop("_metadata"))

# PHASE 3: BRONZE ENGINE
def bronze_engine(spark, config):
    table_name = config['table_bronze']
    source_file = config['source_file']
    checkpoint_path = config['checkpoint_path']
    run_batch_id = config['batch_id_bronze']
    full_schema = f"{config['schema']}, _corrupt_record STRING"

    batch_table = config['log_table']['batch']
    sla_table = config['log_table']['sla']

    batch_schema = "batch_id STRING, layer STRING, status STRING, processed_at TIMESTAMP"
    sla_schema = "batch_id STRING, layer STRING, total_rows LONG, clean_rows LONG, quarantine_rows LONG, status STRING, processed_at TIMESTAMP"

    spark.sql(f"CREATE TABLE IF NOT EXISTS {batch_table} ({batch_schema}) USING DELTA")
    spark.sql(f"CREATE TABLE IF NOT EXISTS {sla_table} ({sla_schema}) USING DELTA")

    logger.info(f"[START] Starting Bronze Auto Loader Batch: {run_batch_id}")

    upsert_log(spark, batch_table, batch_schema,
               [(run_batch_id, "BRONZE", "STARTED", datetime.now())], ["batch_id", "layer"])
    
    try:
        # STEP 1: NEW BATCH > TRANSFORM > WRITE > CALCULATE SLA > LOG SUCCESS
        def new_batch(stream_df, stream_id):
            if stream_df.isEmpty():
                return
            
            df_final = audit_col(stream_df, run_batch_id)

            try:
                dt = DeltaTable.forName(spark, table_name)
                condition = "t.txn_id = s.txn_id AND t._ingest_date = s._ingest_date"
                (dt.alias("t").merge(df_final.alias("s"), condition)
                 .whenMatchedUpdateAll()
                 .whenNotMatchedInsertAll()
                 .execute())
                
            except Exception as e:
                if "DELTA_MISSING_DELTA_TABLE" in str(e) or "NOT_FOUND" in str(e) or "does not exist" in str(e):
                    (df_final.write.format("delta")
                        .mode("overwrite")
                        .option("overwriteSchema", "true")
                        .partitionBy("_ingest_date")
                        .saveAsTable(table_name))
                else:
                    raise e

            df_res = (spark.table(table_name)
                        .filter(F.col("_batch_id_bronze") == run_batch_id)
                        .groupBy("_record_status")
                        .count()
                        .collect())
            
            stats = {r["_record_status"] : r["count"] for r in df_res}
            t_rows = sum(stats.values())
            c_rows = stats.get("CLEAN", 0)
            q_rows = stats.get("CORRUPT", 0)

            upsert_log(spark, sla_table, sla_schema,
               [(run_batch_id, "BRONZE", t_rows, c_rows, q_rows, "SUCCESS", datetime.now())], ["batch_id", "layer"])

        # STEP 2: READ & WRITE STREAM
        df_stream = (spark.readStream.format("cloudFiles")
                        .option("cloudFiles.format", "json")
                        .option("cloudFiles.schemaLocation", f"{checkpoint_path}/schema")
                        .option("cloudFiles.useIncrementalListing", "true")
                        .schema(full_schema)
                        .load(source_file)
                        .select("*", "_metadata"))
        
        query = (df_stream.writeStream
                    .foreachBatch(new_batch)
                    .option("checkpointLocation", f"{checkpoint_path}/data")
                    .trigger(availableNow=True)
                    .start())
        
        query.awaitTermination()

        # STEP 3: SKIPPED / SUCCESS LOG
        df_batch_table = spark.table(batch_table).filter(F.col("batch_id") == run_batch_id).select("status").collect()
        current_status = df_batch_table[0]['status'] if df_batch_table else "UNKNOWN"

        if current_status == "STARTED":
            df_sla_table = spark.table(sla_table).filter(F.col("batch_id") == run_batch_id).collect()

            upsert_log(spark, batch_table, batch_schema,
                       [(run_batch_id, "BRONZE", "SUCCESS", datetime.now())], ["batch_id", "layer"])
            
            if df_sla_table:
                row = df_sla_table[0]
                t_rows, c_rows, q_rows = row['total_rows'], row['clean_rows'], row['quarantine_rows']
                
                logger.info(f"[SUCCESS] Bronze completed. Total: {t_rows}, Clean: {c_rows}, Quarantine: {q_rows}")

            else:
                upsert_log(spark, batch_table, batch_schema,
                        [(run_batch_id, "BRONZE", "SKIPPED", datetime.now())], ["batch_id", "layer"])
                logger.info(f"[SKIP] No new data found. Logging as SKIPPED")
        
    except Exception as e:
        upsert_log(spark, batch_table, batch_schema,
               [(run_batch_id, "BRONZE", "FAILED", datetime.now())], ["batch_id", "layer"])
        upsert_log(spark, sla_table, sla_schema,
               [(run_batch_id, "BRONZE", 0, 0, 0, "FAILED", datetime.now())], ["batch_id", "layer"])
        logger.error(f"[ERROR] Bronze failed: {e}")
        raise e

# PHASE 4: RUN ENGINE
if __name__ == "__main__":
    bronze_engine(spark, bronze_config)